### Here we're storing Raw data only in Bronze_Source Tables and Bronze_destination tables will have all complete data


##### The distination tables of entities have upsert implemented and thosse of events have appen only implemented for incremental loading.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
files = ['allergies.csv', 'encounters.csv', 'observations.csv', 'payer_transitions.csv', 'careplans.csv', 'imaging_studies.csv', 'organizations.csv', 'procedures.csv', 'conditions.csv', 'immunizations.csv', 'patients.csv', 'providers.csv', 'devices.csv', 'medications.csv', 'payers.csv', 'supplies.csv']

entities = ['patients', 'providers', 'organizations', 'payers']
entity_pk = {"patients": "Id","providers": "Id","organizations": "Id","payers": "Id"}

In [0]:
for file in files:    
    raw_df = spark.read.csv(f'/Volumes/syntheaproject/default/syntheavolume/{file}', header=True, inferSchema=True)
    raw_df.createOrReplaceTempView(f"source_{file[:-4]}")
    processed_df = (raw_df.withColumns({"ingested_at": F.current_timestamp(), "sourcefile": F.lit(file[:-4])}))

    destTable = f"syntheaproject.bronze.{file[:-4]}"
    if file[:-4] in entities:
        entity_df = (processed_df.withColumns({"is_active" :  F.lit(True), "updated_at" : F.lit(None).cast("timestamp")}))
        if not spark.catalog.tableExists(destTable):
            entity_df.write.format("delta").mode("overwrite").saveAsTable(destTable)
        else:
            primary_key = entity_pk[file[:-4]]
            entity_df.createOrReplaceTempView("incoming_updates")
            spark.sql(f'merge into {destTable} AS target USING incoming_updates AS source ON target.{primary_key} = source.{primary_key} AND target.is_active = true when matched then update set target.is_active = false, target.updated_at = current_timestamp()')
            entity_df.write.format("delta").mode("append").saveAsTable(destTable)
    else :# for events table
        processed_df.write.format("delta").mode("append").saveAsTable(destTable)

###### Implementing CDF (Changing data feed) based incremental pull in silver source

In [0]:
%sql
select * from syntheaproject.bronze.patients limit 1

In [0]:
%sql
select * from syntheaproject.bronze.encounters limit 1